In [ ]:
#@title Mount Drive
from google.colab import drive
drive.mount('/content/drive')
data_path = '/content/drive/MyDrive/Research/Euphemism Attention/' #@param {type: "string"}

In [ ]:
#@title Data/Model Config
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset
import pandas as pd

csv_name = 'EN-euphs-2025-processed.csv' #@param {type: "string"}
data_column = 'text_raw' #@param {type: "string"}
bounded = False #@param {type: "boolean"}
test_size = 0.2 #@param {type: "number"}
model_name = 'FacebookAI/xlm-roberta-base' #@param {type: "string"}

CSV = data_path + "datasets/" + csv_name

data = pd.read_csv(CSV)
print(f"Dataset loaded from '{CSV}'")

train_df, eval_df = train_test_split(data, test_size=test_size, random_state=42)
# train_df = data
# eval_df = data
print(f"Training data size: {len(train_df)}")
print(f"Evaluation data size: {len(eval_df)}")

tok_delim = 'Ġ' if model_name == 'FacebookAI/roberta-base' else '▁'

BOUND_TOKEN = '[PET_BOUNDARY]'
TOKENIZER = AutoTokenizer.from_pretrained(model_name)
if bounded: TOKENIZER.add_special_tokens({'additional_special_tokens': [BOUND_TOKEN]})

def init_model(output_attentions=False):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        output_attentions=output_attentions,
        attn_implementation="eager"
    )
    if bounded: model.resize_token_embeddings(len(TOKENIZER))
    return model

def tokenize(input):
    # max_length (OOM)
    return TOKENIZER(input, max_length=512, padding='max_length', truncation=True, return_tensors='pt')

def get_tokens(tokenized_input):
    tokens = TOKENIZER.convert_ids_to_tokens(tokenized_input['input_ids'][0].tolist())
    seq_len = (tokenized_input['attention_mask'] == 1).sum().item()
    return tokens[:seq_len]

class EuphemismDataset(Dataset):
    def __init__(self, data):
        tokenized_data = [tokenize(text) for text in data[data_column].tolist()]
        encoded_labels = LabelEncoder().fit_transform(data['label'])
        self.input_ids = torch.cat([item['input_ids'] for item in tokenized_data], dim=0)
        self.attention_mask = torch.cat([item['attention_mask'] for item in tokenized_data], dim=0)
        self.labels = torch.tensor(encoded_labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

In [ ]:
#@title Model Finetuning (L4 GPU)
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os
import gc

frozen_layers = 7 #@param {type: "integer"}
epochs = 20 #@param {type: "integer"}
batch_size = 32 #@param {type: "integer"}
learn_rate = 2e-5 #@param {type: "number"}

# Clear memory between runs
if 'trainer' in globals(): del trainer
if 'finetuned_model' in globals(): del finetuned_model
gc.collect()
torch.cuda.empty_cache()


# Define compute_metrics function
def compute_metrics(pred):
    labels = pred.label_ids
    # Handle case where predictions might be a tuple
    logits = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    preds = logits.argmax(-1)

    # Set zero_division=0.0 to avoid the UndefinedMetricWarning
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0.0) # Assuming binary classification
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

data_collator = DataCollatorWithPadding(tokenizer=TOKENIZER, padding=True)

# Initialize with output_attentions=False for training to save massive amounts of VRAM
finetuned_model = init_model(output_attentions=False)
# finetuned_model.gradient_checkpointing_enable() # OOM

# Freeze lower N layers
for i, layer in enumerate(finetuned_model.roberta.encoder.layer):
    if i < frozen_layers:
        for param in layer.parameters():
            param.requires_grad = False

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
finetuned_model.to(device)
print(f"Model moved to: {device}")

args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=epochs,
    learning_rate=learn_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    report_to='none',
    bf16=True, # Highly recommended for L4/A100 GPUs!
    dataloader_num_workers=4
)

trainer = Trainer(
    model=finetuned_model,
    args=args,
    train_dataset=EuphemismDataset(train_df),
    eval_dataset=EuphemismDataset(eval_df),
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

trainer.train()

finetuned_model.config.output_attentions = True
print("Training complete. Attentions re-enabled for inference/plotting.")

In [ ]:
#@title Save Model (Overwrites Old Files)
import os

save_path = data_path + 'finetuned_euph_model'

finetuned_model.save_pretrained(save_path)
TOKENIZER.save_pretrained(save_path)
train_df.to_csv(os.path.join(save_path, 'split_train.csv'), index=False)
eval_df.to_csv(os.path.join(save_path, 'split_eval.csv'), index=False)

print(f"Model, tokenizer, and splits saved to: {save_path}")